# Offline Policy Evaluation


### Introduction

This notebook demonstrates the use of offline policy evaluation for MABs.

### Objectives

#### Evaluation:

Evaluate the performance of a MAB using multiple offline policy estimators.

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

from pybandits.cmab import CmabBernoulliCC
from pybandits.offline_policy_evaluator import OfflinePolicyEvaluator

%load_ext autoreload
%autoreload 2

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Generate data

We first generate a binarly labeled data set, with a two dimensional feature space, and is not lineraly seprabale.
We then split the data set to a training data setm and a test data set.

In [2]:
n_samples = 1000
n_actions = 2
n_batches = 3
n_rewards = 1
n_groups = 2
n_features = 3

In [3]:
unique_actions = [f"a{i}" for i in range(n_actions)]
action_ids = np.random.choice(unique_actions, n_samples * n_batches)
batches = [i for i in range(n_batches) for _ in range(n_samples)]
rewards = [np.random.randint(2, size=(n_samples * n_batches)) for _ in range(n_rewards)]
action_true_rewards = {(a, r): np.random.rand() for a in unique_actions for r in range(n_rewards)}
true_rewards = [
    np.array([action_true_rewards[(a, r)] for a in action_ids]).reshape(n_samples * n_batches) for r in range(n_rewards)
]
groups = np.random.randint(n_groups, size=n_samples * n_batches)
action_costs = {action: np.random.rand() for action in unique_actions}
costs = np.array([action_costs[a] for a in action_ids])
context = np.random.rand(n_samples * n_batches, n_features)
action_propensity_score = {action: np.random.rand() for action in unique_actions}
propensity_score = np.array([action_propensity_score[a] for a in action_ids])
df = pd.DataFrame(
    {
        "batch": batches,
        "action_id": action_ids,
        "cost": costs,
        "group": groups,
        **{f"reward_{r}": rewards[r] for r in range(n_rewards)},
        **{f"true_reward_{r}": true_rewards[r] for r in range(n_rewards)},
        **{f"context_{i}": context[:, i] for i in range(n_features)},
        "propensity_score": propensity_score,
    }
)
contextual_features = [col for col in df.columns if col.startswith("context")]

## Generate Model

Using the cold_start method of CmabBernoulliCC, we can create a model to be used for offline policy evaluation.

In [4]:
action_ids_cost = {action_id: df["cost"][df["action_id"] == action_id].iloc[0] for action_id in unique_actions}

mab = CmabBernoulliCC.cold_start(action_ids_cost=action_ids_cost, n_features=len(contextual_features))

## OPE

Given the model and the OPE data from the logging policy, we can either evaluate the model using the logging policy, or update it with the logging policy data prior to the evaluation.

In [5]:
evaluator = OfflinePolicyEvaluator(
    logged_data=df,
    split_prop=0.5,
    n_trials=10,
    fast_fit=True,
    scaler=MinMaxScaler(),
    ope_estimators=None,
    verbose=True,
    propensity_score_model_type="batch_empirical",
    expected_reward_model_type="gbm",
    importance_weights_model_type="logreg",
    batch_feature="batch",
    action_feature="action_id",
    reward_feature="reward_0",
    true_reward_feature="true_reward_0",
    contextual_features=contextual_features,
    group_feature="group",
    cost_feature="cost",
    propensity_score_feature="propensity_score",
)

  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:00<00:00, 291.93it/s]


2026-02-08 06:52:12.054 | INFO     | pybandits.offline_policy_evaluator:_estimate_propensity_score:853 - Data batch-empirical estimation of propensity score.


2026-02-08 06:52:12.063 | INFO     | pybandits.offline_policy_evaluator:_estimate_expected_reward:904 - Data prediction of expected reward based on gbm model.


In [6]:
evaluator.evaluate(mab=mab, visualize=True, n_mc_experiments=1000)

2026-02-08 06:52:12.372 | INFO     | pybandits.offline_policy_evaluator:estimate_policy:1001 - Data prediction of expected policy based on Monte Carlo experiments using 4 cores.


  0%|          | 0/1000 [00:00<?, ?it/s]

2026-02-08 06:52:12.408 | INFO     | pybandits.offline_policy_evaluator:_mab_predict_serialized:1286 - Predicting actions for MC experiment 1.


2026-02-08 06:52:12.409 | INFO     | pybandits.offline_policy_evaluator:_mab_predict_serialized:1286 - Predicting actions for MC experiment 3.


2026-02-08 06:52:12.408 | INFO     | pybandits.offline_policy_evaluator:_mab_predict_serialized:1286 - Predicting actions for MC experiment 2.


...

2026-02-08 06:52:40.061 | INFO     | pybandits.offline_policy_evaluator:_estimate_importance_weight:943 - Data prediction of importance weights based on logreg model.


2026-02-08 06:52:40.139 | INFO     | pybandits.offline_policy_evaluator:evaluate:1089 - Offline Policy Evaluation for reward_0.


/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/scipy/stats/_resampling.py:147: RuntimeWarning: invalid value encountered in scalar divide
  a_hat = 1/6 * sum(nums) / sum(dens)**(3/2)
/home/runner/work/pybandits/pybandits/pybandits/offline_policy_estimator.py:145: DegenerateDataWarning: The BCa confidence interval cannot be calculated. This problem is known to occur when the distribution is degenerate or the statistic is np.min.
  bootstrap_result = bootstrap(


Loading BokehJS ...

,value,lower,upper,std,estimator,objective
0,0.497260,0.464989,0.530372,0.016673,b-ipw,reward_0
1,0.496253,0.490835,0.502007,0.002885,dm,reward_0
2,0.496217,0.464856,0.528421,0.016257,dr,reward_0
3,0.496253,0.490805,0.501968,0.002854,dros-opt,reward_0
4,0.496217,0.464337,0.527752,0.016333,dros-pess,reward_0
5,0.497228,0.464569,0.530725,0.016667,ipw,reward_0
6,0.000000,NaN,NaN,0.000000,rep,reward_0
7,0.496217,0.464857,0.528104,0.016166,sndr,reward_0
8,0.497660,0.465023,0.530367,0.016696,snips,reward_0
9,0.496217,0.464780,0.528804,0.016219,sg-dr,reward_0


In [7]:
evaluator.update_and_evaluate(mab=mab, visualize=True, n_mc_experiments=1000)

2026-02-08 06:52:41.348 | INFO     | pybandits.offline_policy_evaluator:_update_mab:1171 - Offline policy update for <class 'pybandits.cmab.CmabBernoulliCC'>.


/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/pytensor/link/c/cmodule.py:2968: UserWarning: PyTensor could not link to a BLAS installation. Operations that might benefit from BLAS will be severely degraded.
This usually happens when PyTensor is installed via pip. We recommend it be installed via conda/mamba/pixi instead.
Alternatively, you can use an experimental backend such as Numba or JAX that perform their own BLAS optimizations, by setting `pytensor.config.mode == 'NUMBA'` or passing `mode='NUMBA'` when compiling a PyTensor function.
For more options and details see https://pytensor.readthedocs.io/en/latest/troubleshooting.html#how-do-i-configure-test-my-blas-library
  warnings.warn(


/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/rich/live.py:260: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/rich/live.py:260: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

2026-02-08 06:52:48.592 | INFO     | pybandits.offline_policy_evaluator:estimate_policy:1001 - Data prediction of expected policy based on Monte Carlo experiments using 4 cores.


...

100%|██████████| 1000/1000 [00:32<00:00, 31.09it/s]

2026-02-08 06:53:20.958 | INFO     | pybandits.offline_policy_evaluator:_estimate_importance_weight:943 - Data prediction of importance weights based on logreg model.


2026-02-08 06:53:21.047 | INFO     | pybandits.offline_policy_evaluator:evaluate:1089 - Offline Policy Evaluation for reward_0.


Loading BokehJS ...

,value,lower,upper,std,estimator,objective
0,0.501453,0.450701,0.555133,0.026557,b-ipw,reward_0
1,0.494541,0.489144,0.500268,0.002829,dm,reward_0
2,0.496852,0.454219,0.538449,0.021814,dr,reward_0
3,0.494541,0.488986,0.500192,0.002841,dros-opt,reward_0
4,0.496852,0.454033,0.539989,0.021957,dros-pess,reward_0
5,0.502494,0.452549,0.557692,0.026550,ipw,reward_0
6,0.230769,0.076923,0.615385,0.133651,rep,reward_0
7,0.496854,0.453745,0.538747,0.021805,sndr,reward_0
8,0.502780,0.453186,0.556098,0.026126,snips,reward_0
9,0.496852,0.454297,0.538366,0.021595,sg-dr,reward_0
